# WESAD Stress Detection — Training Pipeline
Closed-Loop Music Therapy Framework — AI Inference Engine

This notebook trains the stress classifier used by the Streamlit app:
1. Load WESAD (or synthetic fallback data for dev/testing)
2. Window + extract HRV/EDA statistical features
3. Train a RandomForest baseline
4. Train a 1D CNN-LSTM hybrid deep model
5. Compare Accuracy / F1 and save the production model files

> **Note:** run `pip install -r requirements.txt` first. Real WESAD data goes
> in `data/WESAD/S2/S2.pkl`, etc. Without it, this notebook automatically
> uses synthetic physiological data so you can validate the pipeline
> end-to-end before the real download finishes.

In [ ]:
import sys, os
sys.path.append("..")  # so we can import data_pipeline.py, train_model.py from project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_pipeline import load_wesad_or_synthetic, load_wesad_dataset, make_windows
from train_model import subject_wise_split, train_rf_baseline, build_cnn_lstm, train_cnn_lstm

np.random.seed(42)

## 1. Load data
Tries real WESAD first; falls back to synthetic dev data with a printed warning.

In [ ]:
WESAD_ROOT = "../data/WESAD"  # point this at your downloaded dataset

try:
    data = load_wesad_dataset(WESAD_ROOT)
    source = "real"
except FileNotFoundError:
    data, source = load_wesad_or_synthetic(WESAD_ROOT)

print("Data source:", source)
print("Total samples:", len(data["ecg"]))

## 2. Windowing + feature engineering
20s windows, 50% overlap, HRV (mean HR, std HR, RMSSD) + EDA (mean, max, std).

In [ ]:
X, X_seq, y, feature_names, groups = make_windows(
    data["ecg"], data["eda"], data["labels"], subject_id=data["subject_id"]
)

print("Tabular features:", X.shape)
print("Raw sequences (for CNN-LSTM):", X_seq.shape)
print("Labels:", y.shape, "| class balance:", np.bincount(y))
print("Feature names:", feature_names)

pd.DataFrame(X, columns=feature_names).describe()

### Sanity check: feature distributions by class

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
df_feat = pd.DataFrame(X, columns=feature_names)
df_feat["label"] = y
for ax, feat in zip(axes.flat, feature_names):
    df_feat.boxplot(column=feat, by="label", ax=ax)
    ax.set_title(feat)
    ax.set_xlabel("0=baseline, 1=stress")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 3. Subject-wise train/test split
Splitting by subject (not window) avoids leakage and gives an honest F1 estimate.

In [ ]:
X_train, X_test, y_train, y_test, train_idx, test_idx = subject_wise_split(X, y, groups)
X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
print("Train windows:", len(y_train), "| Test windows:", len(y_test))

## 4. RandomForest baseline
Fast, interpretable, works under compute constraints per the brief.

In [ ]:
rf_model, scaler, rf_metrics = train_rf_baseline(X_train, y_train, X_test, y_test)
rf_metrics

## 5. 1D CNN-LSTM hybrid deep model
Captures spatial sensor patterns (Conv1D) + temporal dependencies (LSTM) across the raw windowed ECG/EDA sequences.

> Requires TensorFlow. If it isn't installed in your environment, `pip install tensorflow` (or `tensorflow-macos` + `tensorflow-metal` on Apple Silicon) then re-run this cell.

In [ ]:
cnn_model, norm_stats, cnn_metrics = train_cnn_lstm(
    X_seq_train, y_train, X_seq_test, y_test, epochs=30
)
cnn_metrics

## 6. Compare models

In [ ]:
import json
comparison = pd.DataFrame({
    "RandomForest": rf_metrics,
    "CNN-LSTM": cnn_metrics,
}).T
comparison

## 7. Save production artifacts
Both are saved so `model_loader.py` can pick whichever is available at runtime
(CNN-LSTM preferred, RF as automatic fallback).

In [ ]:
import pickle

os.makedirs("../models", exist_ok=True)

with open("../models/rf_baseline.pkl", "wb") as f:
    pickle.dump({"model": rf_model, "scaler": scaler, "feature_names": feature_names}, f)

cnn_model.save("../models/cnn_lstm_stress.h5")
with open("../models/cnn_lstm_norm_stats.pkl", "wb") as f:
    pickle.dump(norm_stats, f)

with open("../models/metrics.json", "w") as f:
    json.dump({"data_source": source, "random_forest": rf_metrics, "cnn_lstm": cnn_metrics}, f, indent=2)

print("Saved: rf_baseline.pkl, cnn_lstm_stress.h5, cnn_lstm_norm_stats.pkl, metrics.json")

## Notes for the paper
- **Subject-wise split** is used deliberately (`GroupShuffleSplit` on subject IDs) — a random
  window-level split inflates accuracy because adjacent windows from the same subject/session
  are highly correlated. Report the subject-wise numbers.
- **RMSSD** is computed from R-peak-derived RR intervals via `scipy.signal.find_peaks`, not a
  library like NeuroKit2 — worth stating as a limitation/method choice in your Methods section.
- Re-run this notebook against the **real WESAD dataset** (not the synthetic fallback) before
  reporting final numbers. The printed `source` variable tells you which one was used.
